## IMAGES

### Check for Georeferenced Images (Requires Input path)


In [2]:
import os
import re
import warnings
import csv
from pathlib import Path
from collections import defaultdict, Counter

import win32com.client
import pythoncom
import rasterio
import numpy as np

warnings.filterwarnings('ignore')


class GeorefImageProcessor:
    def __init__(self, source_path, output_dir):
        self.source_path = Path(source_path)
        self.output_dir = Path(output_dir)
        self.output_file = "Raster_ImageExtent.csv"

        self.shortcuts = []                 # every .lnk found
        self.shortcuts_with_targets = []    # subset whose target resolved
        self.raster_data = []               # one record per shortcut (final)
        self.broken_targets = []            # lnk -> no/empty target

        print(f"Initialized GeorefImageProcessor")
        print(f"Source path: {self.source_path}")
        print(f"Output directory: {self.output_dir}")

    # ------------------------------------------------------------------ #
    # Parsing
    # ------------------------------------------------------------------ #
    def parse_image_info(self, filename):
        filename = (filename.replace('.lnk', '')
                            .replace(' - Verknüpfung', '')
                            .replace('_Finalized', ''))
        parts = [p.strip() for p in filename.split('#') if p.strip()]

        image_num = flight_num = sortie = bestell_id = date = None

        if parts:
            image_num = parts[0]

        for part in parts:
            if part.startswith('ID') and any(c.isdigit() for c in part[2:]):
                numbers = re.findall(r'\d+', part)
                bestell_id = numbers[0] if numbers else None
                break

        for part in parts:
            if re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                date = part
                break

        if len(parts) >= 3:
            for part in parts[1:-1]:
                if part != bestell_id and not re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                    if not flight_num:
                        flight_num = part
                    elif not sortie:
                        sortie = part

        return {
            'image_number': image_num,
            'flight_number': flight_num,
            'sortie': sortie,
            'bestell_id': bestell_id,
            'date': date,
            'original_name': filename,
        }

    # ------------------------------------------------------------------ #
    # Step 1 – scan for .lnk files
    # ------------------------------------------------------------------ #
    def find_and_process_shortcuts(self):
        print(f"\nScanning for shortcut files in {self.source_path}...")

        for root, dirs, files in os.walk(self.source_path):
            for file in files:
                if file.lower().endswith('.lnk'):
                    file_without_ext = file[:-4]
                    parsed_info = self.parse_image_info(file_without_ext)
                    parsed_info['full_path'] = Path(root) / file
                    parsed_info['folder'] = root
                    self.shortcuts.append(parsed_info)

        print(f"Found {len(self.shortcuts)} shortcut files")
        return self.shortcuts

    # ------------------------------------------------------------------ #
    # Step 2 – resolve shortcut targets
    # ------------------------------------------------------------------ #
    def get_shortcut_target(self, lnk_path):
        try:
            pythoncom.CoInitialize()
            shell = win32com.client.Dispatch("WScript.Shell")
            shortcut = shell.CreateShortCut(str(lnk_path))
            target_path = shortcut.TargetPath
            pythoncom.CoUninitialize()
            return target_path if target_path else None
        except Exception as e:
            print(f"Error reading shortcut {lnk_path}: {e}")
            return None

    def find_modified_versions(self, original_path):
        if not original_path or not original_path.exists():
            return []

        folder = original_path.parent
        stem = original_path.stem
        modified_patterns = ['_modified', '_modifiziert', '_georeferenced', '_processed']
        modified_files = []

        for pattern in modified_patterns:
            potential_file = folder / f"{stem}{pattern}{original_path.suffix}"
            if potential_file.exists():
                modified_files.append(potential_file)

        for file in folder.iterdir():
            if file.is_file() and original_path.suffix == file.suffix:
                file_stem = file.stem
                if (file_stem.startswith(stem) and
                        any(pattern in file_stem for pattern in modified_patterns)):
                    modified_files.append(file)

        return list(set(modified_files))

    def extract_shortcut_targets(self):
        print("\nExtracting shortcut targets and finding modified versions...")

        for shortcut_data in self.shortcuts:
            lnk_path = shortcut_data['full_path']
            target_path = self.get_shortcut_target(lnk_path)

            if not target_path:
                # Keep the record but mark it as broken
                shortcut_data.update({
                    'target_path': None,
                    'target_exists': False,
                    'target_folder': None,
                    'target_filename': '',
                    'target_image_number': None,
                    'target_bestell_id': None,
                    'target_sortie': None,
                    'target_date': None,
                    'broken': True,
                })
                self.broken_targets.append(shortcut_data)
                self.shortcuts_with_targets.append(shortcut_data)
                continue

            target_path = Path(target_path)
            shortcut_data.update({
                'target_path': target_path,
                'target_exists': target_path.exists(),
                'target_folder': target_path.parent,
                'target_filename': target_path.name,
                'broken': False,
            })

            target_info = self.parse_image_info(target_path.stem)
            shortcut_data.update({
                'target_image_number': target_info['image_number'],
                'target_bestell_id': target_info['bestell_id'],
                'target_sortie': target_info['sortie'],
                'target_date': target_info['date'],
            })

            if shortcut_data['target_exists']:
                modified_files = self.find_modified_versions(target_path)
                if modified_files:
                    shortcut_data['modified_files'] = modified_files

            self.shortcuts_with_targets.append(shortcut_data)

        print(f"Successfully processed {len(self.shortcuts_with_targets)} shortcuts")
        print(f"Broken / unresolvable targets: {len(self.broken_targets)}")
        return self.shortcuts_with_targets

    # ------------------------------------------------------------------ #
    # Step 3 – read TIFF metadata
    # ------------------------------------------------------------------ #
    def calculate_tiff_area_extent(self, tiff_path):
        try:
            with rasterio.open(tiff_path) as src:
                rows, cols = src.height, src.width
                transform = src.transform
                pixel_res_x, pixel_res_y = transform.a, -transform.e
                bounds = src.bounds

                pixel_area = abs(pixel_res_x * pixel_res_y)
                total_area = pixel_area * rows * cols

                return {
                    'area_m2': float(total_area),
                    'rows': int(rows),
                    'cols': int(cols),
                    'pixel_res_x_m': float(pixel_res_x),
                    'pixel_res_y_m': float(pixel_res_y),
                    'bounds': bounds,
                    'is_georeferenced': src.crs is not None,
                }
        except Exception as e:
            print(f"Error processing TIFF {tiff_path.name}: {e}")
            return None

    # ------------------------------------------------------------------ #
    # Step 4 – build ONE record per shortcut, with status
    # ------------------------------------------------------------------ #
    def process_georeferenced_images(self):
        print("\nProcessing georeferenced TIFF files...")

        for shortcut in self.shortcuts_with_targets:
            bestell_id = shortcut.get('bestell_id') or shortcut.get('target_bestell_id')
            bestell_id_float = float(bestell_id) if bestell_id and bestell_id.isdigit() else 0.0

            base_record = {
                'BestellID': bestell_id_float,
                'SORTIE': shortcut.get('sortie') or shortcut.get('target_sortie') or '',
                'Bildnummer': shortcut.get('image_number') or shortcut.get('target_image_number') or '',
                'Rows': 0.0,
                'Cols': 0.0,
                'Res_m': 0.0,
                'Area_sqm': 0.0,
                'X_min': 0.0,
                'Y_min': 0.0,
                'X_max': 0.0,
                'Y_max': 0.0,
                'Has_Georef': False,
                'Image_Name': shortcut.get('target_filename', ''),
                'Status': 'OK',
                'LNK_Name': shortcut['full_path'].name,
                'LNK_Path': str(shortcut['full_path']),
                'Target_Path': str(shortcut['target_path']) if shortcut['target_path'] else '',
            }

            # -------- Reason 1: broken / unresolved target -----------
            if shortcut.get('broken') or not shortcut.get('target_path'):
                base_record['Status'] = 'Broken target'
                self.raster_data.append(base_record)
                continue

            # -------- Reason 2: target exists on disk? ---------------
            if not shortcut.get('target_exists'):
                base_record['Status'] = 'Target file missing'
                self.raster_data.append(base_record)
                continue

            # -------- Reason 3: no _modified / _georeferenced sibling
            if 'modified_files' not in shortcut:
                base_record['Status'] = 'No modified/georeferenced file'
                self.raster_data.append(base_record)
                continue

            # -------- Try every modified sibling ---------------------
            found_any = False
            for modified_file in shortcut['modified_files']:
                if modified_file.suffix.lower() not in ['.tif', '.tiff']:
                    # remember it in case nothing else succeeds
                    if not found_any:
                        base_record['Status'] = f'Modified file not TIFF ({modified_file.name})'
                    continue

                area_info = self.calculate_tiff_area_extent(modified_file)

                if area_info is None:
                    if not found_any:
                        base_record['Status'] = f'Unreadable TIFF ({modified_file.name})'
                    continue

                if not area_info['is_georeferenced']:
                    if not found_any:
                        base_record['Status'] = f'No CRS ({modified_file.name})'
                    continue

                # Success -> emit a record for this georef file
                found_any = True
                record = base_record.copy()
                record.update({
                    'Rows': float(area_info['rows']),
                    'Cols': float(area_info['cols']),
                    'Res_m': float((area_info['pixel_res_x_m'] + area_info['pixel_res_y_m']) / 2),
                    'Area_sqm': float(area_info['area_m2']),
                    'X_min': float(area_info['bounds'].left),
                    'Y_min': float(area_info['bounds'].bottom),
                    'X_max': float(area_info['bounds'].right),
                    'Y_max': float(area_info['bounds'].top),
                    'Has_Georef': True,
                    'Image_Name': modified_file.name,
                    'Status': 'OK',
                })
                self.raster_data.append(record)

            if not found_any and base_record['Status'] == 'OK':
                base_record['Status'] = 'No usable georeferenced TIFF'
                self.raster_data.append(base_record)

        print(f"Total records produced: {len(self.raster_data)}")
        return self.raster_data

    # ------------------------------------------------------------------ #
    # Step 5 – mark duplicates (informational only)
    # ------------------------------------------------------------------ #
    def annotate_duplicates(self):
        keys = Counter(
            (r['BestellID'], r['Bildnummer'], r['Image_Name'])
            for r in self.raster_data
        )
        for r in self.raster_data:
            key = (r['BestellID'], r['Bildnummer'], r['Image_Name'])
            if keys[key] > 1:
                r['Status'] = (r['Status'] + ' | DUPLICATE KEY').strip(' |')
        dupes = {k: v for k, v in keys.items() if v > 1}
        print(f"\nDuplicate (BestellID, Bildnummer, Image_Name) keys: {len(dupes)}")
        for k, v in list(dupes.items())[:10]:
            print(f"  {v}x  {k}")

    # ------------------------------------------------------------------ #
    # Step 6 – print table
    # ------------------------------------------------------------------ #
    def display_results_table(self):
        print("\n" + "=" * 140)
        print("COMPLETE IMAGE LIST (one row per .lnk, with status)")
        print("=" * 140)

        header = (f"{'No.':<4} {'BestellID':<10} {'SORTIE':<10} {'Bildnummer':<12} "
                  f"{'Rows':<6} {'Cols':<6} {'Res(m)':<8} {'Area_sqm':<12} "
                  f"{'X_min':<10} {'Y_min':<10} {'X_max':<10} {'Y_max':<10} "
                  f"{'Georef':<7} {'Status'}")
        print(header)
        print("-" * 140)

        for i, r in enumerate(self.raster_data):
            georef = 'OK' if r['Has_Georef'] else 'MISSING'
            print(f"{i+1:<4} "
                  f"{r['BestellID']:<10.0f} "
                  f"{r['SORTIE']:<10} "
                  f"{r['Bildnummer']:<12} "
                  f"{r['Rows']:<6.0f} "
                  f"{r['Cols']:<6.0f} "
                  f"{r['Res_m']:<8.2f} "
                  f"{r['Area_sqm']:<12.0f} "
                  f"{r['X_min']:<10.2f} "
                  f"{r['Y_min']:<10.2f} "
                  f"{r['X_max']:<10.2f} "
                  f"{r['Y_max']:<10.2f} "
                  f"{georef:<7} "
                  f"{r['Status']}")

        return self.raster_data

    # ------------------------------------------------------------------ #
    # Step 7 – CSV
    # ------------------------------------------------------------------ #
    def save_to_csv(self, data):
        output_path = self.output_dir / self.output_file
        self.output_dir.mkdir(parents=True, exist_ok=True)
        print(f"\nSaving results to CSV: {output_path}")

        fieldnames = ['BestellID', 'SORTIE', 'Bildnummer', 'Rows', 'Cols', 'Res_m',
                      'Area_sqm', 'X_min', 'Y_min', 'X_max', 'Y_max',
                      'Has_Georef', 'Image_Name', 'Status',
                      'LNK_Name', 'LNK_Path', 'Target_Path']

        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for record in data:
                writer.writerow({k: record.get(k, '') for k in fieldnames})

        print(f"CSV saved successfully with {len(data)} records")
        return output_path

    # ------------------------------------------------------------------ #
    # Step 8 – summary
    # ------------------------------------------------------------------ #
    def display_summary_statistics(self):
        print("\n" + "=" * 80)
        print("SUMMARY STATISTICS")
        print("=" * 80)

        total = len(self.raster_data)
        georef = sum(1 for r in self.raster_data if r['Has_Georef'])
        missing = total - georef

        print(f"Total shortcuts found:            {len(self.shortcuts)}")
        print(f"Total records in output:          {total}")
        print(f"Georeferenced images found:       {georef}")
        print(f"Missing georeferenced versions:   {missing}")

        print("\nReasons for missing georeferenced versions:")
        reasons = Counter(
            r['Status'].split(' | ')[0]
            for r in self.raster_data if not r['Has_Georef']
        )
        for reason, count in reasons.most_common():
            print(f"  {count:>5}  {reason}")

        if georef > 0:
            areas = [r['Area_sqm'] for r in self.raster_data if r['Has_Georef']]
            print(f"\nGeoreferenced area statistics:")
            print(f"  Total area:   {sum(areas):,.0f} m² ({sum(areas)/10000:,.2f} ha)")
            print(f"  Average area: {np.mean(areas):,.0f} m² ({np.mean(areas)/10000:,.2f} ha)")
            print(f"  Min area:     {min(areas):,.0f} m²")
            print(f"  Max area:     {max(areas):,.0f} m²")

            georef_records = [r for r in self.raster_data if r['Has_Georef']]
            xs_min = [r['X_min'] for r in georef_records]
            xs_max = [r['X_max'] for r in georef_records]
            ys_min = [r['Y_min'] for r in georef_records]
            ys_max = [r['Y_max'] for r in georef_records]
            print(f"\nGeographic extent ranges:")
            print(f"  X: {min(xs_min):.4f} → {max(xs_max):.4f}")
            print(f"  Y: {min(ys_min):.4f} → {max(ys_max):.4f}")

    # ------------------------------------------------------------------ #
    # Main
    # ------------------------------------------------------------------ #
    def run(self):
        print("\n" + "=" * 80)
        print("GEOREF IMAGE PROCESSOR – STARTING")
        print("=" * 80)

        self.find_and_process_shortcuts()
        self.extract_shortcut_targets()
        self.process_georeferenced_images()
        self.annotate_duplicates()

        all_data = self.display_results_table()
        csv_path = self.save_to_csv(all_data)
        self.display_summary_statistics()

        print(f"\nProcessing completed successfully!")
        print(f"Output saved to: {csv_path}")

        return {
            'shortcuts_count': len(self.shortcuts),
            'records_count': len(all_data),
            'georef_count': sum(1 for r in all_data if r['Has_Georef']),
            'output_path': csv_path,
        }


if __name__ == "__main__":
    SOURCE_PATH = r"D:\TemporaryID"
    OUTPUT_DIR = r"G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck"

    processor = GeorefImageProcessor(SOURCE_PATH, OUTPUT_DIR)
    results = processor.run()

Initialized GeorefImageProcessor
Source path: D:\TemporaryID
Output directory: G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck

GEOREF IMAGE PROCESSOR – STARTING

Scanning for shortcut files in D:\TemporaryID...
Found 232 shortcut files

Extracting shortcut targets and finding modified versions...
Successfully processed 232 shortcuts
Broken / unresolvable targets: 0

Processing georeferenced TIFF files...
Total records produced: 233

Duplicate (BestellID, Bildnummer, Image_Name) keys: 0

COMPLETE IMAGE LIST (one row per .lnk, with status)
No.  BestellID  SORTIE     Bildnummer   Rows   Cols   Res(m)   Area_sqm     X_min      Y_min      X_max      Y_max      Georef  Status
--------------------------------------------------------------------------------------------------------------------------------------------
1    1767       106G_2874  3147         11532  13088  0.21     6822617      518448.19  5393073.34 521230.85  5395525.17 OK      OK
2    1767       

### Quality control through GCP file 

In [3]:
import os
import os
import sys
import numpy as np
from osgeo import gdal
from pathlib import Path
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# Windows shortcut support
try:
    import win32com.client
    HAS_WIN32COM = True
except ImportError:
    HAS_WIN32COM = False
    print("⚠️ win32com not available – .lnk resolution disabled")


# ============================================================================
# VALIDATOR CLASS
# ============================================================================

class GeoreferencingValidator:

    def __init__(self, root_folder):
        self.root_folder = Path(root_folder)

        if not self.root_folder.exists():
            raise FileNotFoundError(f"Folder does not exist: {self.root_folder}")

        self.all_results = []
        self.image_summary_stats = []
        self.shortcut_map = []
        self.current_jobs = []
        
        # NEW: Track special cases
        self.images_with_no_points = []
        self.images_with_zero_residuals = []
        self.images_with_nan_residuals = []

        # Configurable thresholds
        self.warning_threshold = 10  # px - flag for inspection
        self.critical_threshold = 20  # px - definitely needs inspection
        self.min_gcps_for_confidence = 4  # Minimum GCPs for reliable georeferencing
        
        # NEW: Threshold for "essentially zero" residuals
        self.zero_threshold = 1e-10  # Very small threshold for scientific notation zeros

    # ----------------------------------------------------------------------
    # SHORTCUT RESOLUTION
    # ----------------------------------------------------------------------

    def resolve_shortcut(self, lnk_path):
        if not HAS_WIN32COM:
            return None

        try:
            shell = win32com.client.Dispatch("WScript.Shell")
            shortcut = shell.CreateShortCut(str(lnk_path))
            target = Path(shortcut.Targetpath)
            return target
        except Exception as e:
            print(f"❌ Error resolving shortcut {lnk_path.name}: {e}")
            return None

    # ----------------------------------------------------------------------
    # FIND SHORTCUTS → IMAGE MAP
    # ----------------------------------------------------------------------

    def find_shortcutted_images(self):
        """
        Returns a list of dicts:
        {
            shortcut_name,
            image_name,
            target_folder
        }
        """
        image_jobs = []

        for lnk in self.root_folder.rglob("*.lnk"):
            print(f"\nProcessing shortcut: {lnk.name}")
            
            target = self.resolve_shortcut(lnk)
            if not target:
                print(f"  ❌ Could not resolve shortcut")
                continue
                
            if not target.exists():
                print(f"  ❌ Target does not exist: {target}")
                continue

            # Get the shortcut name without .lnk extension
            shortcut_name = lnk.stem
            
            # Determine the clean image name based on the target
            if target.is_file() and target.suffix.lower() in ['.tif', '.tiff']:
                # Shortcut points directly to a TIFF file
                clean_image_name = target.stem
                print(f"  ✓ Shortcut points to TIFF: {target.name}")
            else:
                # Shortcut points to a folder or other file type
                # Use the shortcut name and clean it
                clean_image_name = shortcut_name
                
                # Remove common "Verknüpfung" patterns if they exist
                verknupfung_patterns = [
                    " - Verknüpfung_Finalized",
                    " - Verknüpfung",
                    " - Verknüpfung_Finalized.tif",
                    " - Verknüpfung.tif",
                    "_Verknüpfung_Finalized",
                    "_Verknüpfung"
                ]
                
                for pattern in verknupfung_patterns:
                    if pattern in clean_image_name:
                        clean_image_name = clean_image_name.replace(pattern, "")
                        print(f"  ✓ Removed pattern: {pattern}")
                
                # Also remove .tif/.tiff extensions if they're at the end
                if clean_image_name.lower().endswith('.tif'):
                    clean_image_name = clean_image_name[:-4]
                elif clean_image_name.lower().endswith('.tiff'):
                    clean_image_name = clean_image_name[:-5]
                
                clean_image_name = clean_image_name.strip()
            
            print(f"  Cleaned name: '{clean_image_name}'")
            print(f"  Target: {target}")

            image_jobs.append({
                "shortcut_name": lnk.name,
                "image_name": clean_image_name,  # Use cleaned name for .points file search
                "original_shortcut_name": shortcut_name,  # Keep original for reference
                "target_folder": target.parent if target.is_file() else target
            })

        if not image_jobs:
            print("❌ No valid shortcuts found")
        else:
            print(f"\n✓ Found {len(image_jobs)} valid shortcut-referenced images")

        return image_jobs

    # ----------------------------------------------------------------------
    # PROCESS ONE IMAGE ONLY
    # ----------------------------------------------------------------------

    def process_single_image(self, job):
        image_name = job["image_name"]
        original_name = job["original_shortcut_name"]
        folder = job["target_folder"]

        print("\n" + "="*70)
        print(f"PROCESSING IMAGE: {image_name}")
        print(f"Original shortcut: {original_name}")
        print(f"Target folder: {folder}")
        print("="*70)

        # Look for .points files (where residuals are stored)
        points_files = []
        
        # Pattern 1: ImageName.tif.points (most common)
        points_pattern1 = folder / f"{image_name}.tif.points"
        if points_pattern1.exists():
            print(f"  ✓ Found points file: {points_pattern1.name}")
            points_files.append(points_pattern1)
        
        # Pattern 2: ImageName.points (without .tif)
        points_pattern2 = folder / f"{image_name}.points"
        if points_pattern2.exists():
            print(f"  ✓ Found points file: {points_pattern2.name}")
            points_files.append(points_pattern2)
            
        # Pattern 3: ImageName_modified.tif.points
        points_pattern3 = folder / f"{image_name}_modified.tif.points"
        if points_pattern3.exists():
            print(f"  ✓ Found points file: {points_pattern3.name}")
            points_files.append(points_pattern3)
            
        # Pattern 4: Try with original name (without "Verknüpfung" patterns)
        original_cleaned = original_name
        verknupfung_patterns = [" - Verknüpfung_Finalized", " - Verknüpfung"]
        for pattern in verknupfung_patterns:
            if pattern in original_cleaned:
                original_cleaned = original_cleaned.replace(pattern, "").strip()
        
        points_pattern4 = folder / f"{original_cleaned}.tif.points"
        if points_pattern4.exists() and points_pattern4 not in points_files:
            print(f"  ✓ Found points file (from original name): {points_pattern4.name}")
            points_files.append(points_pattern4)
        
        # Pattern 5: Try with .points extension on original cleaned name
        points_pattern5 = folder / f"{original_cleaned}.points"
        if points_pattern5.exists() and points_pattern5 not in points_files:
            print(f"  ✓ Found points file (original name without .tif): {points_pattern5.name}")
            points_files.append(points_pattern5)
            
        # Pattern 6: Search for any .points file containing the image name
        if not points_files:
            print(f"\nSearching for any .points files containing '{image_name}'...")
            for points_file in folder.glob("*.points"):
                if image_name in points_file.name:
                    print(f"  ⚠️  Found (partial match): {points_file.name}")
                    points_files.append(points_file)
        
        # Also check for files in the alignment subfolder
        alignment_folder = folder / "alignment"
        if alignment_folder.exists():
            print(f"\nChecking alignment folder: {alignment_folder}")
            for points_file in alignment_folder.glob("*.points"):
                # Check for matches with both cleaned and original names
                if (image_name in points_file.name or 
                    original_cleaned in points_file.name or 
                    original_name in points_file.name):
                    print(f"  ⚠️  Found in alignment folder: {points_file.name}")
                    points_files.append(points_file)
        
        print(f"\nTotal points files found: {len(points_files)}")
        if points_files:
            for i, points_file in enumerate(points_files, 1):
                print(f"  {i}. {points_file.name}")
        else:
            print(f"❌ NO POINTS FILES FOUND FOR {image_name}")
            # NEW: Track images with no points data
            self.images_with_no_points.append({
                "Image": image_name,
                "Original_Name": original_name,
                "Folder": folder,
                "Reason": "No .points file found"
            })
            return

        image_results_map = {}
        any_points_file_processed = False

        for points_file in points_files:
            print(f"\nProcessing points file: {points_file.name}")
            results = self.process_points_file(points_file, image_name)
            if results:
                any_points_file_processed = True
                key = image_name
                image_results_map.setdefault(key, []).extend(results)
                self.all_results.extend(results)
            else:
                print(f"  ⚠️  No GCP data found in this points file")

        if not any_points_file_processed:
            print(f"❌ No GCP data found in any points file for {image_name}")
            # NEW: Track images with points files but no valid GCP data
            self.images_with_no_points.append({
                "Image": image_name,
                "Original_Name": original_name,
                "Folder": folder,
                "Reason": "Points file exists but contains no GCP data"
            })
            return

        if image_results_map:
            summaries = self.calculate_image_summary_for_folder(folder.name, image_results_map)
            self.image_summary_stats.extend(summaries)
            
            # NEW: Check if this image has zero residuals
            for summary in summaries:
                if summary.get("All_Zero_Residuals", False):
                    self.images_with_zero_residuals.append({
                        "Image": image_name,
                        "Original_Name": original_name,
                        "Folder": folder,
                        "Max_Residual": summary["Max_Residual_px"],
                        "Avg_Residual": summary["Avg_Residual_px"]
                    })
        else:
            print(f"❌ No valid GCP data found for {image_name}")
            self.images_with_no_points.append({
                "Image": image_name,
                "Original_Name": original_name,
                "Folder": folder,
                "Reason": "Points files found but no valid GCP data extracted"
            })

    # ----------------------------------------------------------------------
    # PROCESS POINTS FILE (where residuals are actually stored)
    # ----------------------------------------------------------------------

    def process_points_file(self, points_path, image_name):
        results = []

        print(f"\nProcessing points file: {points_path.name}")
        
        try:
            # Read the points file with latin-1 encoding (based on your output)
            with open(points_path, 'r', encoding='latin-1') as f:
                content = f.read()
            
            # Split into lines
            lines = content.strip().split('\n')
            
            # Find the data section (skip header lines)
            data_start = 0
            header_line = ""
            for i, line in enumerate(lines):
                if line.startswith('mapX,mapY,sourceX,sourceY,enable,dX,dY,residual'):
                    header_line = line
                    data_start = i + 1
                    break
            
            if data_start == 0:
                # Try alternative header
                for i, line in enumerate(lines):
                    if line.startswith('mapX,mapY,pixelX,pixelY,enable,dX,dY,residual'):
                        header_line = line
                        data_start = i + 1
                        break
            
            if data_start == 0:
                print(f"  ⚠️  Could not find data header in points file")
                print(f"  First few lines of the file:")
                for i, line in enumerate(lines[:5]):
                    print(f"    Line {i}: {line}")
                return results
            
            print(f"  Found header: {header_line}")
            print(f"  Found {len(lines) - data_start} GCP entries")
            
            # Track if all residuals are essentially zero
            all_essentially_zero_residuals = True
            
            # Process each GCP entry
            for i, line in enumerate(lines[data_start:], 1):
                if not line.strip():
                    continue
                    
                try:
                    # Split the line by commas
                    parts = line.strip().split(',')
                    
                    if len(parts) < 8:
                        print(f"  ⚠️  Line {i}: Not enough columns, skipping")
                        continue
                    
                    # Extract values - be more robust about column positions
                    try:
                        # Try to parse dX, dY, residual from expected positions
                        dX = float(parts[5])      # Residual in X (pixels)
                        dY = float(parts[6])      # Residual in Y (pixels)
                        residual = float(parts[7])  # Total residual (pixels)
                    except (ValueError, IndexError):
                        # If that fails, try to find columns by content
                        print(f"  ⚠️  Line {i}: Could not parse from fixed positions, trying alternative parsing")
                        print(f"     Line: {line}")
                        print(f"     Parts: {parts}")
                        
                        # Try to find float values in any column
                        for idx, part in enumerate(parts):
                            try:
                                val = float(part)
                                print(f"     Column {idx}: '{part}' -> {val}")
                            except:
                                pass
                        
                        # Skip this line if we can't parse it
                        continue
                    
                    # NEW: Check if residual is essentially zero (using scientific notation threshold)
                    if abs(residual) > self.zero_threshold:
                        all_essentially_zero_residuals = False
                    
                    # Determine individual GCP status
                    if residual < 5:
                        gcp_status = "EXCELLENT"
                    elif residual < self.warning_threshold:
                        gcp_status = "GOOD"
                    elif residual < self.critical_threshold:
                        gcp_status = "WARNING"
                    else:
                        gcp_status = "CRITICAL"
                    
                    results.append({
                        "Image_Name": image_name,
                        "GCP_ID": f"GCP_{i:03d}",
                        "Residual_X_px": dX,
                        "Residual_Y_px": dY,
                        "Total_Residual_px": residual,
                        "GCP_Status": gcp_status,
                        "Points_File": points_path.name
                    })
                    
                except ValueError as e:
                    print(f"  ⚠️  Line {i}: Error parsing line - {e}")
                    print(f"     Line content: {line}")
                    print(f"     Parts: {parts}")
                    continue
            
            print(f"  ✓ Successfully processed {len(results)} GCPs")
            
            # If all residuals are essentially zero, show more debug info
            if all_essentially_zero_residuals and len(results) > 0:
                print(f"  ⚠️  WARNING: All {len(results)} GCPs have essentially zero residuals (<{self.zero_threshold})")
                print(f"  Sample residual values:")
                for i, result in enumerate(results[:3]):
                    print(f"    GCP_{i+1}: {result['Total_Residual_px']}")
                if len(results) > 3:
                    print(f"    ... and {len(results)-3} more")
                
        except Exception as e:
            print(f"  ❌ Error processing points file: {e}")
            import traceback
            traceback.print_exc()
            
        return results

    # ----------------------------------------------------------------------
    # IMAGE CLASSIFICATION
    # ----------------------------------------------------------------------

    def calculate_image_summary_for_folder(self, folder_name, image_results_map):
        summaries = []

        for image, results in image_results_map.items():
            residuals = [r["Total_Residual_px"] for r in results]
            gcp_statuses = [r["GCP_Status"] for r in results]
            
            # Check if all residuals are essentially zero
            all_essentially_zero = all(abs(r) < self.zero_threshold for r in residuals)
            
            # Count GCPs by status
            excellent_count = gcp_statuses.count("EXCELLENT")
            good_count = gcp_statuses.count("GOOD")
            warning_count = gcp_statuses.count("WARNING")
            critical_count = gcp_statuses.count("CRITICAL")
            
            # Calculate statistics
            max_residual = max(residuals) if residuals else 0
            avg_residual = np.mean(residuals) if residuals else 0
            rmse = np.sqrt(np.mean(np.square(residuals))) if residuals else 0
            
            # Determine overall image status
            status = self.classify_image_quality(
                total_gcps=len(results),
                critical_count=critical_count,
                warning_count=warning_count,
                max_residual=max_residual,
                avg_residual=avg_residual,
                rmse=rmse,
                all_essentially_zero=all_essentially_zero
            )
            
            # Determine inspection priority
            priority = self.determine_inspection_priority(status, max_residual, critical_count, all_essentially_zero)
            
            summaries.append({
                "Image": image,
                "Status": status,
                "Priority": priority,
                "Total_GCPs": len(results),
                "Critical_GCPs": critical_count,
                "Warning_GCPs": warning_count,
                "Max_Residual_px": round(max_residual, 12),  # Show more decimal places for small values
                "Avg_Residual_px": round(avg_residual, 12),
                "RMSE_px": round(rmse, 12),
                "All_Zero_Residuals": all_essentially_zero,
                "Residual_Sample": residuals[:3] if residuals else []  # Sample of residual values
            })

            # Print summary with special warning for essentially zero residuals
            if all_essentially_zero and len(results) > 0:
                print(f"\n⚠️  {image}: {status} ({priority} priority) - ALL RESIDUALS ARE ESSENTIALLY ZERO!")
                print(f"  ⚠️  Residual values: {residuals[:3]}...")
                print(f"  ⚠️  Max: {max_residual:.12f}px, Avg: {avg_residual:.12f}px, RMSE: {rmse:.12f}px")
                print(f"  GCPs: {len(results)} total")
            else:
                print(f"\n📊 {image}: {status} ({priority} priority)")
                print(f"  Max: {max_residual:.2f}px, Avg: {avg_residual:.2f}px, RMSE: {rmse:.2f}px")
                print(f"  GCPs: {len(results)} total, {critical_count} critical, {warning_count} warnings")

        return summaries

    def classify_image_quality(self, total_gcps, critical_count, warning_count, 
                              max_residual, avg_residual, rmse, all_essentially_zero):
        """
        Simple classification based on residuals
        """
        # Rule 0: All residuals are essentially zero
        if all_essentially_zero:
            return "ZERO_RESIDUALS"
        
        # Rule 1: Not enough GCPs
        if total_gcps < self.min_gcps_for_confidence:
            return "INSUFFICIENT_GCPs"
        
        # Rule 2: Any critical GCPs
        if critical_count > 0:
            return "FAIL"
        
        # Rule 3: Too many warnings (>50% of GCPs)
        if warning_count > 0 and warning_count / total_gcps > 0.5:
            return "NEEDS_INSPECTION"
        
        # Rule 4: High RMSE indicates systematic error
        if rmse > 15:
            return "FAIL"
        
        # Rule 5: High max residual
        if max_residual > 15:
            return "NEEDS_INSPECTION"
        
        # Rule 6: Moderate issues
        if warning_count > 0 or avg_residual > 8:
            return "NEEDS_INSPECTION"
        
        # Default: Pass
        return "PASS"

    def determine_inspection_priority(self, status, max_residual, critical_count, all_essentially_zero):
        """
        Determine inspection priority
        """
        if status == "ZERO_RESIDUALS":
            return "VERY_LOW"  # These need special attention but not for quality issues
        
        if status == "PASS":
            return "LOW"
        elif status == "INSUFFICIENT_GCPs":
            return "MEDIUM"
        elif critical_count > 0:
            if max_residual > 30:
                return "CRITICAL"
            else:
                return "HIGH"
        elif status == "NEEDS_INSPECTION":
            return "MEDIUM"
        else:  # FAIL
            return "HIGH"

    # ----------------------------------------------------------------------
    # DISPLAY RESULTS IN CONSOLE
    # ----------------------------------------------------------------------

    def display_results(self):
        print("\n" + "="*80)
        print("GEOREFERENCING VALIDATION RESULTS")
        print("="*80)
        
        # NEW: Display images with no points data
        if self.images_with_no_points:
            print(f"\n🚫 IMAGES WITH NO POINTS DATA ({len(self.images_with_no_points)} images)")
            print("-"*60)
            for img in self.images_with_no_points:
                print(f"• {img['Image']}")
                print(f"  Original: {img['Original_Name']}")
                print(f"  Folder: {img['Folder']}")
                print(f"  Reason: {img['Reason']}")
                print()
        
        # NEW: Display images with zero residuals
        if self.images_with_zero_residuals:
            print(f"\n🔍 IMAGES WITH ESSENTIALLY ZERO RESIDUALS ({len(self.images_with_zero_residuals)} images)")
            print("-"*60)
            print("These images have residuals < 1e-10 (scientific notation zeros)")
            print("This usually indicates:")
            print("  1. Already georeferenced/rectified images")
            print("  2. Perfect mathematical fit (unlikely in practice)")
            print("  3. Points file format differences")
            print()
            for img in self.images_with_zero_residuals:
                print(f"• {img['Image']}")
                print(f"  Original: {img['Original_Name']}")
                print(f"  Folder: {img['Folder']}")
                print(f"  Max Residual: {img['Max_Residual']:.12f} px")
                print()
        
        # Display GCP details if available
        if self.all_results:
            print("\n📋 DETAILED GCP ANALYSIS")
            print("-"*40)
            
            # Group results by image
            image_groups = {}
            for result in self.all_results:
                image_name = result["Image_Name"]
                if image_name not in image_groups:
                    image_groups[image_name] = []
                image_groups[image_name].append(result)
            
            # Display each image's GCPs
            for image_name, gcp_results in image_groups.items():
                print(f"\n📷 {image_name}:")
                print(f"{'-'*30}")
                
                # Check if all residuals are essentially zero for this image
                all_essentially_zero = all(abs(g["Total_Residual_px"]) < self.zero_threshold for g in gcp_results)
                if all_essentially_zero and len(gcp_results) > 0:
                    print(f"⚠️  ALL RESIDUALS ARE ESSENTIALLY ZERO (<{self.zero_threshold})")
                
                # Display GCP table
                print(f"{'GCP':<8} {'Residual (px)':<25} {'Status':<15}")
                print(f"{'-'*55}")
                
                for gcp in gcp_results:
                    gcp_id = gcp["GCP_ID"]
                    residual = gcp["Total_Residual_px"]
                    status = gcp["GCP_Status"]
                    
                    # Format residual with scientific notation for very small values
                    if abs(residual) < 0.001:
                        residual_display = f"{residual:.12e}"
                    else:
                        residual_display = f"{residual:.4f}"
                    
                    # Add status indicators
                    if status == "CRITICAL":
                        status_display = f"🔴 {status}"
                    elif status == "WARNING":
                        status_display = f"🟡 {status}"
                    elif status == "GOOD":
                        status_display = f"🟢 {status}"
                    else:
                        status_display = f"✅ {status}"
                    
                    # Highlight essentially zero residuals
                    if abs(residual) < self.zero_threshold:
                        residual_display = f"{residual_display:<25} ⚠️"
                    else:
                        residual_display = f"{residual_display:<25}"
                    
                    print(f"{gcp_id:<8} {residual_display} {status_display:<15}")
        
        # Display image summary if available
        if self.image_summary_stats:
            print("\n\n📊 IMAGE VALIDATION SUMMARY")
            print("="*80)
            
            # Display summary table
            print(f"\n{'Image':<40} {'Status':<25} {'Priority':<15} {'Max Res (px)':<20} {'GCPs':<8}")
            print(f"{'-'*115}")
            
            for summary in self.image_summary_stats:
                image = summary["Image"]
                status = summary["Status"]
                priority = summary["Priority"]
                max_res = summary["Max_Residual_px"]
                total_gcps = summary["Total_GCPs"]
                all_essentially_zero = summary.get("All_Zero_Residuals", False)
                
                # Add priority indicators
                if priority == "CRITICAL":
                    priority_display = f"🔴 {priority}"
                elif priority == "HIGH":
                    priority_display = f"🟠 {priority}"
                elif priority == "MEDIUM":
                    priority_display = f"🟡 {priority}"
                elif priority == "VERY_LOW":
                    priority_display = f"⚪ {priority}"
                else:
                    priority_display = f"🟢 {priority}"
                
                # Add status indicators
                if status == "ZERO_RESIDUALS":
                    status_display = f"🔵 ZERO_RESIDUALS"
                elif status == "FAIL":
                    status_display = f"🔴 {status}"
                elif status == "NEEDS_INSPECTION":
                    status_display = f"🟡 {status}"
                elif status == "INSUFFICIENT_GCPs":
                    status_display = f"⚫ {status}"
                else:
                    status_display = f"✅ {status}"
                
                # Format max residual for display
                if abs(max_res) < 0.001:
                    max_res_display = f"{max_res:.12e}"
                else:
                    max_res_display = f"{max_res:.4f}"
                
                # Highlight zero residuals
                if all_essentially_zero:
                    max_res_display = f"{max_res_display:<20} ⚠️"
                else:
                    max_res_display = f"{max_res_display:<20}"
                
                print(f"{image:<40} {status_display:<25} {priority_display:<15} {max_res_display} {total_gcps:<8}")
            
            # Display overall statistics
            self.display_overall_statistics()
        
        # Display classification guidelines
        self.display_classification_guidelines()
        
        print("\n" + "="*80)
        print("VALIDATION COMPLETE")
        print("="*80)
        
        # NEW: Final summary of special cases
        print("\n📈 FINAL SUMMARY")
        print("-"*40)
        print(f"Total images processed: {len(self.current_jobs)}")
        print(f"Images with valid GCP data: {len(self.image_summary_stats)}")
        print(f"Images with no points data: {len(self.images_with_no_points)}")
        print(f"Images with essentially zero residuals: {len(self.images_with_zero_residuals)}")
        
        if self.images_with_no_points:
            print(f"\n⚠️  ACTION NEEDED: Check these {len(self.images_with_no_points)} images for missing .points files")
        
        if self.images_with_zero_residuals:
            print(f"\n🔍 INSPECT: {len(self.images_with_zero_residuals)} images have essentially zero residuals")
            print("   These may already be georeferenced or need manual verification")

    def display_overall_statistics(self):
        """Display overall statistics about the validation"""
        if not self.image_summary_stats:
            return
        
        print(f"\n\n📈 OVERALL STATISTICS")
        print("-"*40)
        
        # Count images by status
        status_counts = {}
        priority_counts = {}
        zero_residual_count = 0
        
        for summary in self.image_summary_stats:
            status = summary["Status"]
            priority = summary["Priority"]
            all_zero = summary.get("All_Zero_Residuals", False)
            
            status_counts[status] = status_counts.get(status, 0) + 1
            priority_counts[priority] = priority_counts.get(priority, 0) + 1
            if all_zero:
                zero_residual_count += 1
        
        # Calculate percentages
        total_images = len(self.image_summary_stats)
        
        print(f"\n📁 Images with GCP data: {total_images}")
        print(f"📊 Total GCPs Analyzed: {len(self.all_results)}")
        print(f"🔍 Images with essentially zero residuals: {zero_residual_count}")
        
        if zero_residual_count > 0:
            print(f"   ⚠️  These images have residuals < {self.zero_threshold}")
            print(f"   Possible causes: already georeferenced, perfect fit, or format issue")
        
        print(f"\n📊 STATUS DISTRIBUTION:")
        for status, count in sorted(status_counts.items()):
            percentage = (count / total_images) * 100
            print(f"  {status:<25} : {count:>3} images ({percentage:>5.1f}%)")
        
        print(f"\n🎯 PRIORITY DISTRIBUTION:")
        priority_order = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "VERY_LOW"]
        for priority in priority_order:
            if priority in priority_counts:
                count = priority_counts[priority]
                percentage = (count / total_images) * 100
                print(f"  {priority:<25} : {count:>3} images ({percentage:>5.1f}%)")
        
        # Calculate average metrics (excluding images with all essentially zero residuals)
        non_zero_images = [s for s in self.image_summary_stats if not s.get("All_Zero_Residuals", False)]
        
        if non_zero_images:
            avg_max_residual = np.mean([s["Max_Residual_px"] for s in non_zero_images])
            avg_rmse = np.mean([s["RMSE_px"] for s in non_zero_images])
            
            print(f"\n📐 AVERAGE METRICS (excluding {zero_residual_count} images with essentially zero residuals):")
            print(f"  Average Max Residual: {avg_max_residual:.4f} px")
            print(f"  Average RMSE: {avg_rmse:.4f} px")
        else:
            print(f"\n📐 AVERAGE METRICS: All images have essentially zero residuals")

    def display_classification_guidelines(self):
        """Display the quality classification and priority assignment guidelines"""
        print("\n\n📚 QUALITY CLASSIFICATION GUIDELINES")
        print("="*80)
        
        print("\n🎯 QUALITY CLASSIFICATION")
        print("-"*30)
        print(f"• 🔵 ZERO_RESIDUALS: All residuals < {self.zero_threshold}")
        print("• ✅ PASS: Good quality (<10px max residual)")
        print("• ⚠️  NEEDS_INSPECTION: Moderate issues (10-20px residuals)")
        print("• 🔴 FAIL: Critical issues (>20px residuals)")
        print("• ⚫ INSUFFICIENT_GCPs: Too few control points (<4 GCPs)")
        
        print("\n📊 GCP CLASSIFICATION")
        print("-"*30)
        print("• ✅ EXCELLENT: <5px residual")
        print("• 🟢 GOOD: 5-10px residual")
        print("• 🟡 WARNING: 10-20px residual")
        print("• 🔴 CRITICAL: >20px residual")
        
        print("\n🎯 PRIORITY ASSIGNMENT")
        print("-"*30)
        print("• 🔴 CRITICAL: Max residual >30px or multiple critical GCPs")
        print("• 🟠 HIGH: Critical GCPs present")
        print("• 🟡 MEDIUM: Needs inspection but less urgent")
        print("• 🟢 LOW: Minor issues or passes")
        print("• ⚪ VERY_LOW: Essentially zero residuals (already georeferenced?)")
        
        print("\n📝 INTERPRETATION:")
        print("-"*30)
        print("• Images with 🔴 CRITICAL or 🟠 HIGH priority need immediate attention")
        print("• Images with 🟡 MEDIUM priority should be reviewed when possible")
        print("• Images with 🟢 LOW priority are acceptable for most applications")
        print(f"• Images with ⚪ VERY_LOW priority have residuals < {self.zero_threshold}")
        
        print("\n⚠️  SPECIAL NOTES:")
        print("-"*30)
        print(f"• Images with ALL residuals < {self.zero_threshold} may already be georeferenced")
        print("• Check if these images need manual verification")
        print("• Zero residuals can indicate perfect mathematical fit (rare in practice)")

    # ----------------------------------------------------------------------
    # MAIN RUN
    # ----------------------------------------------------------------------

    def run_validation(self):
        print("GEOREFERENCING VALIDATION")
        print("="*60)
        print(f"Root folder: {self.root_folder}")
        print("="*60)
        
        jobs = self.find_shortcutted_images()
        self.current_jobs = jobs  # Store for reference
        
        if not jobs:
            print("❌ No valid shortcut jobs to process")
            return
            
        print(f"\nStarting validation of {len(jobs)} images...")
        
        for i, job in enumerate(jobs, 1):
            print(f"\n{'='*60}")
            print(f"Progress: {i}/{len(jobs)}")
            self.process_single_image(job)

        print(f"\n{'='*60}")
        print("VALIDATION PROCESSING COMPLETE")
        print("="*60)
        
        # Display results instead of generating CSV files
        self.display_results()


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    # Update this path to your actual folder
    MAIN_FOLDER = r"D:\TemporaryID"

    try:
        validator = GeoreferencingValidator(MAIN_FOLDER)
        validator.run_validation()
    except FileNotFoundError as e:
        print(f"❌ ERROR: {e}")
        print("Please check that the folder path is correct.")
    except Exception as e:
        print(f"❌ UNEXPECTED ERROR: {e}")
        import traceback
        traceback.print_exc()

    print("\n" + "="*60)
    print("SCRIPT FINISHED")
    print("="*60)
    input("Press Enter to exit...")

GEOREFERENCING VALIDATION
Root folder: D:\TemporaryID

Processing shortcut: 3015#00347#US7_0140_A#ID1767#09_03_1945.tif.lnk
  ✓ Shortcut points to TIFF: 3015#00347#US7_0140_A#ID1767#09_03_1945.tif
  Cleaned name: '3015#00347#US7_0140_A#ID1767#09_03_1945'
  Target: W:\1945_03_09#00347#US7_0140_A\3015#00347#US7_0140_A#ID1767#09_03_1945.tif

Processing shortcut: 7152#_#US7_3952#ID1767#17_02_1945.tif.lnk
  ✓ Shortcut points to TIFF: 7152#_#US7_3952#ID1767#17_02_1945.tif
  Cleaned name: '7152#_#US7_3952#ID1767#17_02_1945'
  Target: W:\1945_02_17#_#US7_3952\7152#_#US7_3952#ID1767#17_02_1945.tif

Processing shortcut: 1016#00039#D_0039#ID1767#18_02_1943.tif.lnk
  ✓ Shortcut points to TIFF: 1016#00039#D_0039#ID1767#18_02_1943.tif
  Cleaned name: '1016#00039#D_0039#ID1767#18_02_1943'
  Target: W:\1943_02_18#00039#D_0039\1016#00039#D_0039#ID1767#18_02_1943.tif

Processing shortcut: 1017#00039#D_0039#ID1767#18_02_1943.tif.lnk
  ✓ Shortcut points to TIFF: 1017#00039#D_0039#ID1767#18_02_1943.tif
  C

## DATABASE

### Check for Frame Areas, Bewölkung Info (Requires Input Path)

In [5]:
import os
import re
from pathlib import Path
import psycopg2
import pandas as pd


class PostGISImageProcessor:
    """Process PostGIS images from .lnk files and query the database."""

    def __init__(self, base_path, output_path, db_config):
        self.base_path = Path(base_path)
        self.output_path = Path(output_path)
        self.db_config = db_config
        self.lnk_files = []
        self.image_data = []
        self.results_df = None

    # ------------------------------------------------------------------ #
    # File discovery / parsing
    # ------------------------------------------------------------------ #
    def get_lnk_files(self, project_path=None):
        """Find all .lnk files under ``project_path`` (or ``base_path``)."""
        search_root = Path(project_path) if project_path else self.base_path
        self.lnk_files = []
        for root, _, files in os.walk(search_root):
            for file in files:
                if file.lower().endswith('.lnk'):
                    self.lnk_files.append(Path(root) / file)
        return len(self.lnk_files)

    @staticmethod
    def parse_image_filename(filename):
        """Parse ``ImageNumber#...#SORTIE#...`` style filenames."""
        filename = filename.replace('.lnk', '').replace(' - Verknüpfung', '').strip()
        parts = filename.split('#')

        if len(parts) >= 4:
            image_number = parts[0].strip()
            bestell_id = None
            for part in parts:
                id_match = re.search(r'ID(\d+)', part, re.IGNORECASE)
                if id_match:
                    bestell_id = int(id_match.group(1))
                    break
            sortie = parts[2].strip() if len(parts) > 2 else None
            return {
                'ImageNumber': image_number,
                'BestellID': bestell_id,
                'SORTIE': sortie,
            }
        return None

    def parse_all_filenames(self):
        self.image_data = []
        for lnk_file in self.lnk_files:
            parsed = self.parse_image_filename(lnk_file.name)
            if parsed:
                self.image_data.append(parsed)
        return len(self.image_data)

    # ------------------------------------------------------------------ #
    # Helpers
    # ------------------------------------------------------------------ #
    @staticmethod
    def extract_numbers_from_string(value):
        if not value or pd.isna(value):
            return []
        return [int(num) for num in re.findall(r'\d+', str(value))]

    def get_table_columns(self):
        try:
            conn = psycopg2.connect(**self.db_config)
            cursor = conn.cursor()
            cursor.execute("""
                SELECT column_name
                FROM information_schema.columns
                WHERE table_name = 'bildumrahmungen_aktuell'
                ORDER BY ordinal_position;
            """)
            columns = [row[0] for row in cursor.fetchall()]
            cursor.close()
            conn.close()
            return columns
        except Exception as e:
            print(f"Error getting table columns: {e}")
            return []

    @staticmethod
    def _sanitize_name(name):
        """Make a string safe to use as a Windows filename."""
        name = re.sub(r'[<>:"/\\|?*]', '_', name)
        return name.rstrip(" .") or "Project"

    def get_project_groups(self):
        """
        Return an ordered dict mapping project name -> project folder.

        * If .lnk files sit directly under ``base_path``, the project name is
          ``base_path.name``.
        * Otherwise the first path component *below* ``base_path`` becomes the
          project name (i.e. each immediate subfolder is one project).
        """
        groups = {}
        for lnk in self.base_path.rglob("*.lnk"):
            try:
                rel = lnk.relative_to(self.base_path)
            except ValueError:
                continue
            if len(rel.parts) > 1:
                project_name = rel.parts[0]
                project_folder = self.base_path / project_name
            else:
                project_name = self.base_path.name
                project_folder = self.base_path
            project_name = self._sanitize_name(project_name)
            groups.setdefault(project_name, project_folder)
        return groups

    # ------------------------------------------------------------------ #
    # Database queries
    # ------------------------------------------------------------------ #
    def query_database(self):
        """Query the database for all parsed images and store the results."""
        if not self.image_data:
            self.results_df = pd.DataFrame()
            return self.results_df

        table_columns = self.get_table_columns()
        has_weitere_bestell_id = any('weitere bestell' in c.lower() for c in table_columns)
        has_qualitaet_bild_studis = 'Qualität Bild_Studis' in table_columns

        select_columns = ['"Bestell ID"', 'sortie_ori', '"Bildnummer"', '"Bewölkung"']
        if has_weitere_bestell_id:
            select_columns.append('"weitere Bestell ID"')
        if has_qualitaet_bild_studis:
            select_columns.append('"Qualität Bild_Studis"')

        select_clause = 'SELECT ' + ', '.join(select_columns) + """,
            ROUND(ST_Area(geom)::numeric, 6) AS area_sqm,
            ROUND((ST_XMax(geom) - ST_XMin(geom))::numeric, 6) AS width_m,
            ROUND((ST_YMax(geom) - ST_YMin(geom))::numeric, 6) AS height_m,
            ROUND(ST_XMin(geom)::numeric, 6) AS xmin,
            ROUND(ST_YMin(geom)::numeric, 6) AS ymin,
            ROUND(ST_XMax(geom)::numeric, 6) AS xmax,
            ROUND(ST_YMax(geom)::numeric, 6) AS ymax,
            ROUND(ST_X(ST_Centroid(geom))::numeric, 6) AS centroid_x,
            ROUND(ST_Y(ST_Centroid(geom))::numeric, 6) AS centroid_y
        FROM bildumrahmungen_aktuell
        WHERE {where_clause}
        ORDER BY "Bestell ID", sortie_ori, "Bildnummer"
        """

        all_results = []
        conn = None
        try:
            conn = psycopg2.connect(**self.db_config)
            for img in self.image_data:
                bestell_id = img.get('BestellID')
                sortie = img.get('SORTIE')
                image_number = img.get('ImageNumber')

                if not sortie or not image_number:
                    print(
                        f"Skipping file because SORTIE or Bildnummer is missing: "
                        f"BestellID={bestell_id}, SORTIE={sortie}, Bildnummer={image_number}"
                    )
                    continue

                checks = []
                if bestell_id is not None:
                    checks.append((
                        'Bestell ID + SORTIE + Bildnummer',
                        '("Bestell ID" = %s AND sortie_ori = %s AND "Bildnummer" = %s)',
                        [bestell_id, sortie, image_number],
                    ))
                    if has_weitere_bestell_id:
                        checks.append((
                            'weitere Bestell ID + SORTIE + Bildnummer',
                            '("weitere Bestell ID" LIKE %s AND sortie_ori = %s AND "Bildnummer" = %s)',
                            [f'%{bestell_id}%', sortie, image_number],
                        ))

                checks.append((
                    'SORTIE + Bildnummer',
                    '(sortie_ori = %s AND "Bildnummer" = %s)',
                    [sortie, image_number],
                ))

                for method, where_clause, query_params in checks:
                    query = select_clause.format(where_clause=where_clause)
                    result_df = pd.read_sql_query(query, conn, params=query_params)
                    if not result_df.empty:
                        result_df['Match_Method'] = method
                        result_df['Input_BestellID'] = bestell_id
                        result_df['Input_SORTIE'] = sortie
                        result_df['Input_Bildnummer'] = image_number
                        all_results.append(result_df)
                        break
                else:
                    print(
                        f"No match found for BestellID={bestell_id}, "
                        f"SORTIE={sortie}, Bildnummer={image_number}"
                    )

            self.results_df = (
                pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
            )
            if has_weitere_bestell_id and 'weitere Bestell ID' in self.results_df.columns:
                self.results_df['weitere_Bestell_ID_numbers'] = (
                    self.results_df['weitere Bestell ID']
                    .apply(self.extract_numbers_from_string)
                )
            return self.results_df
        except Exception as e:
            print(f"Database error: {e}")
            self.results_df = pd.DataFrame()
            return self.results_df
        finally:
            if conn is not None:
                conn.close()

    # ------------------------------------------------------------------ #
    # Output construction
    # ------------------------------------------------------------------ #
    def create_output_dataframe(self, project_name=None):
        """Build the formatted output DataFrame from ``self.results_df``."""
        if self.results_df is None or self.results_df.empty:
            return pd.DataFrame()

        output_df = pd.DataFrame()

        if project_name is not None:
            output_df['Project'] = project_name

        output_df['BestellID'] = pd.to_numeric(
            self.results_df['Bestell ID'], errors='coerce'
        ).astype('Int64')
        output_df['SORTIE'] = self.results_df['sortie_ori'].astype(str)
        output_df['Bildnummer'] = self.results_df['Bildnummer'].astype(str)
        output_df['Bewölkung'] = self.results_df['Bewölkung'].fillna('').astype(str)

        if 'Match_Method' in self.results_df.columns:
            output_df['Match_Method'] = self.results_df['Match_Method'].astype(str)
        if 'Input_BestellID' in self.results_df.columns:
            output_df['Input_BestellID'] = self.results_df['Input_BestellID']
        if 'Input_SORTIE' in self.results_df.columns:
            output_df['Input_SORTIE'] = self.results_df['Input_SORTIE'].astype(str)
        if 'Input_Bildnummer' in self.results_df.columns:
            output_df['Input_Bildnummer'] = self.results_df['Input_Bildnummer'].astype(str)

        if 'Qualität Bild_Studis' in self.results_df.columns:
            output_df['Qualität_Bild_Studis'] = (
                self.results_df['Qualität Bild_Studis'].fillna('').astype(str)
            )

        if 'weitere Bestell ID' in self.results_df.columns:
            output_df['weitere_Bestell_ID_raw'] = (
                self.results_df['weitere Bestell ID'].fillna('').astype(str)
            )
        if 'weitere_Bestell_ID_numbers' in self.results_df.columns:
            output_df['weitere_Bestell_ID_numbers'] = (
                self.results_df['weitere_Bestell_ID_numbers']
                .apply(lambda x: ', '.join(map(str, x)) if x else '')
            )

        spatial_columns = {
            'area_sqm': 'Area_sqm',
            'width_m': 'Width_m',
            'height_m': 'Height_m',
            'xmin': 'X_min',
            'ymin': 'Y_min',
            'xmax': 'X_max',
            'ymax': 'Y_max',
            'centroid_x': 'Centroid_X',
            'centroid_y': 'Centroid_Y',
        }
        for src_col, dest_col in spatial_columns.items():
            if src_col in self.results_df.columns:
                output_df[dest_col] = pd.to_numeric(
                    self.results_df[src_col], errors='coerce'
                ).astype('float64')

        return output_df

    def save_to_csv(self, output_df, filename):
        if output_df.empty:
            return None
        self.output_path.mkdir(parents=True, exist_ok=True)
        output_file = self.output_path / filename
        output_df.to_csv(
            output_file, index=False,
            float_format='%.6f', encoding='utf-8-sig',
        )
        return output_file

    def display_results(self, output_df, max_rows=None):
        if output_df.empty:
            print("No results to display")
            return

        print(f"\n{'=' * 120}")
        print(f"COMPLETE IMAGE LIST ({len(output_df)} images)")
        print(f"{'=' * 120}")

        pd.set_option('display.float_format', '{:.6f}'.format)
        pd.set_option('display.max_rows', max_rows or len(output_df))
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', 20)
        print(output_df.to_string(index=False))
        pd.reset_option('display.float_format')
        pd.reset_option('display.max_rows')

    # ------------------------------------------------------------------ #
    # High-level entry points
    # ------------------------------------------------------------------ #
    def process(self, project_path=None, project_name=None, output_filename=None):
        """
        Process a single project (or the entire ``base_path``).

        Returns
        -------
        (output_df, output_file, project_name)
        """
        search_root = Path(project_path) if project_path else self.base_path
        print(f"Processing .lnk files from: {search_root}")

        if project_name is None:
            project_name = search_root.name
        project_name = self._sanitize_name(project_name)

        if output_filename is None:
            output_filename = f"{project_name}_DB_imageExtent.csv"

        # 1) collect + parse
        file_count = self.get_lnk_files(search_root)
        print(f"  Found {file_count} .lnk files")
        parsed_count = self.parse_all_filenames()
        print(f"  Parsed {parsed_count} files")

        if not self.image_data:
            print("  No valid image data to process")
            return None, None, project_name

        # 2) query
        self.query_database()
        if self.results_df.empty:
            print("  No matching records found in database")
            return None, None, project_name

        print(f"  Found {len(self.results_df)} matching records in database")

        # 3) output + save
        output_df = self.create_output_dataframe(project_name=project_name)
        output_file = self.save_to_csv(output_df, output_filename)
        print(f"  Output saved to: {output_file}")

        return output_df, output_file, project_name

    def process_all_projects(self):
        """
        Process every project subfolder under ``base_path``.

        * One CSV per project: ``<ProjectName>_DB_imageExtent.csv``
        * One combined DataFrame returned & displayed at the end
          (with a ``Project`` column).
        """
        project_groups = self.get_project_groups()

        if not project_groups:
            print(f"No .lnk files found under {self.base_path}")
            return None, None

        print(f"Found {len(project_groups)} project(s):")
        for name in project_groups:
            print(f"  - {name}")

        all_outputs = []
        saved_files = []

        for project_name, project_folder in project_groups.items():
            print(f"\n{'#' * 100}")
            print(f"# PROJECT: {project_name}")
            print(f"{'#' * 100}")

            output_df, output_file, _ = self.process(
                project_path=project_folder,
                project_name=project_name,
            )

            if output_df is not None and not output_df.empty:
                all_outputs.append(output_df)
                if output_file is not None:
                    saved_files.append(output_file)

        if not all_outputs:
            print("\nNo results produced for any project.")
            return None, None

        combined_df = pd.concat(all_outputs, ignore_index=True)

        # Display only the final combined list.
        self.display_results(combined_df)

        if 'Match_Method' in combined_df.columns:
            print(f"\n{'=' * 80}")
            print('MATCHING METHOD SUMMARY:')
            print(f"{'=' * 80}")
            print(combined_df['Match_Method'].value_counts().to_string())

        print(f"\n{'=' * 80}")
        print('PER-PROJECT OUTPUT FILES:')
        print(f"{'=' * 80}")
        for f in saved_files:
            print(f"  {f}")

        return combined_df, saved_files


# ========== CONFIGURATION ==========
BASE_PATH = r"D:\TemporaryID"
OUTPUT_PATH = r"G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck\DatabaseOutputs"

DB_CONFIG = {
    'host': '192.168.190.10',
    'port': 5432,
    'database': 'Server_LBA',
    'user': 'LBA_team',
    'password': 'lba',
    'sslmode': 'prefer',
}

# ========== MAIN EXECUTION ==========
if __name__ == "__main__":
    processor = PostGISImageProcessor(
        base_path=BASE_PATH,
        output_path=OUTPUT_PATH,
        db_config=DB_CONFIG,
    )

    combined_df, saved_files = processor.process_all_projects()

    if combined_df is not None:
        print(f"\nProcessing completed successfully!")
        print(f"Total images processed: {len(combined_df)}")
        print(f"Projects: {combined_df['Project'].nunique()}")
        print(f"CSV files written: {len(saved_files)}")

Found 11 project(s):
  - ID1767 Ostfildern-Nellingen - LBDB
  - ID1773 Saarwellingen - LBDB
  - ID1775 Neckarsulm - LBDB
  - ID1780 Löffingen_Talstraße - LBDB
  - ID1781 Cleebronn_Gemarkung - LBDB
  - ID1783 Neckarsulm_2 - LBDB
  - ID1784 Dußlingen_Maltschachstraße - LBDB
  - ID1787 Weil_im_Schönbuch - LBDB
  - ID1788 Schönaich_Gemarkung - LBDB
  - ID1801 Mannheim-Sandhofen - LBDB
  - ID8108 Laufach-Heigenbrücken, Strecke 5200 km 70,8 – 76,5, Einzelflächen - LBDB

####################################################################################################
# PROJECT: ID1767 Ostfildern-Nellingen - LBDB
####################################################################################################
Processing .lnk files from: D:\TemporaryID\ID1767 Ostfildern-Nellingen - LBDB
  Found 26 .lnk files
  Parsed 26 files
  Found 26 matching records in database
  Output saved to: G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck\DatabaseOutputs\ID1767 Ost

### Combining the database csv results

In [6]:
from pathlib import Path
from datetime import datetime
import pandas as pd

INPUT_DIR  = Path(r"G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck\DatabaseOutputs\Neuer Ordner")

# Timestamp for the output filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_CSV = INPUT_DIR / f"ALL_PROJECTS_DB_imageExtent_{timestamp}.csv"

# All per-project CSVs (exclude any merged files)
csv_files = [
    f for f in INPUT_DIR.glob("*_DB_imageExtent.csv")
    if not f.stem.startswith("ALL_PROJECTS_DB_imageExtent")
]

if not csv_files:
    raise SystemExit(f"No *_DB_imageExtent.csv files found in {INPUT_DIR}")

# Read, add a Project column if missing, concatenate
frames = []
for f in sorted(csv_files):
    df = pd.read_csv(f, encoding="utf-8-sig")
    if "Project" not in df.columns:
        df.insert(0, "Project", f.stem.replace("_DB_imageExtent", ""))
    frames.append(df)

merged = pd.concat(frames, ignore_index=True)

merged.to_csv(
    OUTPUT_CSV,
    index=False,
    float_format="%.6f",
    encoding="utf-8-sig",
)

print(f"Merged {len(csv_files)} files -> {merged.shape[0]} rows")
print(f"Saved: {OUTPUT_CSV}")

Merged 11 files -> 232 rows
Saved: G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck\DatabaseOutputs\Neuer Ordner\ALL_PROJECTS_DB_imageExtent_20261008_171257.csv


### Check for Extent Overlap

In [7]:
import pandas as pd
import numpy as np
import os

class ImageOverlapAnalyzer:
    def __init__(self, db_path, raster_path):
        self.db_path = db_path
        self.raster_path = raster_path
        self.db_df = None
        self.raster_df = None
        self.results_df = None
        
        # Column mappings
        self.bestellid_col_db = None
        self.sortie_col_db = None
        self.bildnummer_col_db = None
        self.bestellid_col_raster = None
        self.sortie_col_raster = None
        self.bildnummer_col_raster = None
        self.db_extent_cols = {}
        self.raster_extent_cols = {}
    
    def load_data(self):
        """Load and preprocess CSV files"""
        self.db_df = pd.read_csv(self.db_path)
        self.raster_df = pd.read_csv(self.raster_path)
        
        # Clean column names
        self.db_df = self._clean_column_names(self.db_df)
        self.raster_df = self._clean_column_names(self.raster_df)
        
        # Find key columns
        self._identify_columns()
    
    def _clean_column_names(self, df):
        """Clean column names by removing special characters"""
        df_clean = df.copy()
        df_clean.columns = [col.strip().lower().replace('-', '').replace('_', '').replace('@', '') 
                           for col in df_clean.columns]
        return df_clean
    
    def _find_column(self, df, keywords):
        """Find column based on keywords"""
        for col in df.columns:
            clean_col = col.lower()
            for keyword in keywords:
                if keyword.lower() in clean_col:
                    return col
        return None
    
    def _identify_columns(self):
        """Identify all important columns in both dataframes"""
        # Database columns
        self.bestellid_col_db = self._find_column(self.db_df, ['bestellid', 'bestell', 'orderid'])
        self.sortie_col_db = self._find_column(self.db_df, ['sortie', 'flight', 'mission'])
        self.bildnummer_col_db = self._find_column(self.db_df, ['bildnummer', 'imagenumber', 'imageno', 'bildnr'])
        
        # Raster columns
        self.bestellid_col_raster = self._find_column(self.raster_df, ['bestellid', 'bestell', 'orderid'])
        self.sortie_col_raster = self._find_column(self.raster_df, ['sortie', 'flight', 'mission'])
        self.bildnummer_col_raster = self._find_column(self.raster_df, ['bildnummer', 'imagenumber', 'imageno', 'bildnr'])
        
        # Extent columns
        self.db_extent_cols = self._find_extent_columns(self.db_df)
        self.raster_extent_cols = self._find_extent_columns(self.raster_df)
    
    def _find_extent_columns(self, df):
        """Find extent columns (xmin, xmax, ymin, ymax)"""
        extent_cols = {}
        for col in df.columns:
            if 'xmin' in col:
                extent_cols['xmin'] = col
            elif 'xmax' in col:
                extent_cols['xmax'] = col
            elif 'ymin' in col:
                extent_cols['ymin'] = col
            elif 'ymax' in col:
                extent_cols['ymax'] = col
        return extent_cols
    
    def _calculate_overlap(self, rect1, rect2):
        """Calculate overlap percentage between two rectangles"""
        # rect format: (xmin, ymin, xmax, ymax)
        
        # Calculate intersection
        x_left = max(rect1[0], rect2[0])
        y_bottom = max(rect1[1], rect2[1])
        x_right = min(rect1[2], rect2[2])
        y_top = min(rect1[3], rect2[3])
        
        if x_right < x_left or y_top < y_bottom:
            return 0.0
        
        # Calculate areas
        area_intersection = (x_right - x_left) * (y_top - y_bottom)
        area_rect1 = (rect1[2] - rect1[0]) * (rect1[3] - rect1[1])
        area_rect2 = (rect2[2] - rect2[0]) * (rect2[3] - rect2[1])
        
        if area_rect1 == 0 or area_rect2 == 0:
            return 0.0
        
        # Return average overlap percentage
        overlap1 = (area_intersection / area_rect1) * 100
        overlap2 = (area_intersection / area_rect2) * 100
        return (overlap1 + overlap2) / 2
    
    def _find_db_match(self, raster_row):
        """Find matching database entry for a raster row using SORTIE-first matching"""
        # Get raster data
        raster_bildnummer = raster_row[self.bildnummer_col_raster] if self.bildnummer_col_raster and pd.notna(raster_row[self.bildnummer_col_raster]) else None
        raster_bestellid = raster_row[self.bestellid_col_raster] if self.bestellid_col_raster and pd.notna(raster_row[self.bestellid_col_raster]) else None
        raster_sortie = raster_row[self.sortie_col_raster] if self.sortie_col_raster and pd.notna(raster_row[self.sortie_col_raster]) else None
        
        db_match = None
        
        # FIRST: Match by SORTIE + BestellID + Bildnummer combination
        if all([self.sortie_col_db, self.bestellid_col_db, self.bildnummer_col_db,
                raster_sortie is not None, raster_bestellid is not None, raster_bildnummer is not None]):
            condition = (self.db_df[self.sortie_col_db] == raster_sortie) & \
                       (self.db_df[self.bestellid_col_db] == raster_bestellid) & \
                       (self.db_df[self.bildnummer_col_db] == raster_bildnummer)
            db_matches = self.db_df[condition]
            if not db_matches.empty:
                db_match = db_matches.iloc[0]
        
        # SECOND: If no match, try SORTIE + Bildnummer (without BestellID)
        if db_match is None and self.sortie_col_db and self.bildnummer_col_db:
            if raster_sortie is not None and raster_bildnummer is not None:
                condition = (self.db_df[self.sortie_col_db] == raster_sortie) & \
                           (self.db_df[self.bildnummer_col_db] == raster_bildnummer)
                db_matches = self.db_df[condition]
                if not db_matches.empty:
                    db_match = db_matches.iloc[0]
        
        # THIRD: If still no match, try just SORTIE
        if db_match is None and self.sortie_col_db and self.bildnummer_col_db:
            if raster_sortie is not None and raster_bildnummer is not None:
                condition = (self.db_df[self.sortie_col_db] == raster_sortie)
                db_matches = self.db_df[condition]
                # Look for matching Bildnummer within the same SORTIE
                for _, db_candidate in db_matches.iterrows():
                    if db_candidate[self.bildnummer_col_db] == raster_bildnummer:
                        db_match = db_candidate
                        break
        
        return db_match, raster_bestellid, raster_sortie, raster_bildnummer
    
    def analyze_overlap(self):
        """Main analysis function to calculate overlaps for all images"""
        if self.db_df is None or self.raster_df is None:
            self.load_data()
        
        results = []
        
        # Process each raster image
        for _, raster_row in self.raster_df.iterrows():
            # Find matching database entry
            db_match, raster_bestellid, raster_sortie, raster_bildnummer = self._find_db_match(raster_row)
            
            # Initialize result values
            db_bildnummer = None
            overlap_pct = 0.0
            
            # Calculate overlap if match found
            if db_match is not None:
                db_bildnummer = db_match[self.bildnummer_col_db] if self.bildnummer_col_db and pd.notna(db_match[self.bildnummer_col_db]) else None
                
                # Get extents
                try:
                    db_extent = (
                        float(db_match[self.db_extent_cols['xmin']]),
                        float(db_match[self.db_extent_cols['ymin']]),
                        float(db_match[self.db_extent_cols['xmax']]),
                        float(db_match[self.db_extent_cols['ymax']])
                    )
                    
                    raster_extent = (
                        float(raster_row[self.raster_extent_cols['xmin']]),
                        float(raster_row[self.raster_extent_cols['ymin']]),
                        float(raster_row[self.raster_extent_cols['xmax']]),
                        float(raster_row[self.raster_extent_cols['ymax']])
                    )
                    
                    overlap_pct = self._calculate_overlap(db_extent, raster_extent)
                except (KeyError, ValueError, TypeError, AttributeError):
                    overlap_pct = 0.0
            
            # Add to results
            results.append({
                'BestellID': raster_bestellid,
                'SORTIE': raster_sortie,
                'Raster_Bildnummer': raster_bildnummer,
                'Database_Bildnummer': db_bildnummer,
                'Overlap_%': round(overlap_pct, 2)
            })
        
        # Create results dataframe
        self.results_df = pd.DataFrame(results)
        
        # Sort results
        self._sort_results()
        
        return self.results_df
    
    def _sort_results(self):
        """Sort results by SORTIE, then BestellID, then Raster_Bildnummer"""
        sort_columns = []
        if 'SORTIE' in self.results_df.columns:
            sort_columns.append('SORTIE')
        if 'BestellID' in self.results_df.columns:
            sort_columns.append('BestellID')
        if 'Raster_Bildnummer' in self.results_df.columns:
            sort_columns.append('Raster_Bildnummer')
        
        if sort_columns:
            self.results_df = self.results_df.sort_values(sort_columns, na_position='last').reset_index(drop=True)
    
    def display_results(self):
        """Display analysis results"""
        if self.results_df is None:
            print("No results to display. Please run analyze_overlap() first.")
            return
        
        print("Image Overlap Analysis Results (SORTIE-based matching)")
        print("=" * 80)
        print(f"\nTotal Raster Images: {len(self.results_df)}")
        matched_count = len(self.results_df[self.results_df['Database_Bildnummer'].notna()])
        print(f"Matched with Database: {matched_count} ({matched_count/len(self.results_df)*100:.1f}%)")
        
        print("\nResults Table:")
        print(self.results_df.to_string(index=False))
        print("\n" + "=" * 80)
    
    def display_statistics(self):
        """Display detailed statistics"""
        if self.results_df is None or self.results_df.empty:
            print("No results to analyze.")
            return
        
        matched_count = len(self.results_df[self.results_df['Database_Bildnummer'].notna()])
        
        print("\nMatching Statistics:")
        print(f"  Total images processed: {len(self.results_df)}")
        print(f"  Successfully matched: {matched_count}")
        print(f"  Match rate: {matched_count/len(self.results_df)*100:.1f}%")
        print(f"  Unmatched images: {len(self.results_df) - matched_count}")
        
        # Only show overlap statistics for matched images
        matched_df = self.results_df[self.results_df['Database_Bildnummer'].notna()]
        if not matched_df.empty:
            print("\nOverlap Statistics (matched images only):")
            print(f"  Average Overlap: {matched_df['Overlap_%'].mean():.2f}%")
            print(f"  Maximum Overlap: {matched_df['Overlap_%'].max():.2f}%")
            print(f"  Minimum Overlap: {matched_df['Overlap_%'].min():.2f}%")
            print(f"  Images with 0% overlap: {(matched_df['Overlap_%'] == 0).sum()}")
            print(f"  Images with >90% overlap: {(matched_df['Overlap_%'] > 90).sum()}")
    
    # def save_results(self, output_path=None):
    #     """Save results to CSV file"""
    #     if self.results_df is None:
    #         print("No results to save. Please run analyze_overlap() first.")
    #         return
        
    #     if output_path is None:
    #         output_path = os.path.join(os.path.dirname(self.db_path), "overlap_analysis_results.csv")
        
    #     self.results_df.to_csv(output_path, index=False)
    #     print(f"Results saved to: {output_path}")
    #     return output_path
    
    def get_column_summary(self):
        """Display summary of identified columns"""
        print("Column Identification Summary:")
        print("-" * 40)
        print("Database Columns:")
        print(f"  BestellID: {self.bestellid_col_db}")
        print(f"  SORTIE: {self.sortie_col_db}")
        print(f"  Bildnummer: {self.bildnummer_col_db}")
        print(f"  Extent columns: {self.db_extent_cols}")
        
        print("\nRaster Columns:")
        print(f"  BestellID: {self.bestellid_col_raster}")
        print(f"  SORTIE: {self.sortie_col_raster}")
        print(f"  Bildnummer: {self.bildnummer_col_raster}")
        print(f"  Extent columns: {self.raster_extent_cols}")


# Main execution
if __name__ == "__main__":
    # Define paths
    db_path = r"G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck\DatabaseOutputs\ALL_PROJECTS_DB_imageExtent_20261008_171257.csv"
    raster_path = r"G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck\Raster_ImageExtent.csv"    
    # Create analyzer instance
    analyzer = ImageOverlapAnalyzer(db_path, raster_path)
    
    # Load data
    analyzer.load_data()
    
    # Display column summary
    analyzer.get_column_summary()
    
    # Run analysis
    results = analyzer.analyze_overlap()
    
    # Display results
    analyzer.display_results()
    
    # Display statistics
    analyzer.display_statistics()
    
    # Save results
    # analyzer.save_results()

Column Identification Summary:
----------------------------------------
Database Columns:
  BestellID: bestellid
  SORTIE: sortie
  Bildnummer: bildnummer
  Extent columns: {'xmin': 'xmin', 'ymin': 'ymin', 'xmax': 'xmax', 'ymax': 'ymax'}

Raster Columns:
  BestellID: bestellid
  SORTIE: sortie
  Bildnummer: bildnummer
  Extent columns: {'xmin': 'xmin', 'ymin': 'ymin', 'xmax': 'xmax', 'ymax': 'ymax'}
Image Overlap Analysis Results (SORTIE-based matching)

Total Raster Images: 233
Matched with Database: 233 (100.0%)

Results Table:
 BestellID             SORTIE  Raster_Bildnummer  Database_Bildnummer  Overlap_%
    1780.0        104W_C_0070               4135               4135.0      90.82
    1780.0        104W_C_0070               4136               4136.0      91.38
    1801.0      104W_LIB_0121               3099               3099.0      91.86
    1801.0      104W_LIB_0121               4099               4099.0      93.21
    1801.0      104W_LIB_0121               4100           

## Georeferenced Image cross-check with Basemap

### Resampling

In [8]:
import os
import re
from pathlib import Path
import win32com.client
import pythoncom
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import numpy as np

class ImageResampler:
    def __init__(self, source_path, output_dir, target_pixel_size):
        """
        Initialize ImageResampler.
        
        Args:
            source_path: Path to scan for shortcut files
            output_dir: Directory to save resampled images
            target_pixel_size: Desired pixel size in meters
        """
        self.source_path = Path(source_path)
        self.output_dir = Path(output_dir)
        self.target_pixel_size = target_pixel_size
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Data storage
        self.image_files = []  # Will store paths to georeferenced images
        self.shortcut_data = []  # Will store shortcut and image info
        
    def find_and_parse_shortcuts(self):
        """Find all .lnk files and extract their target paths."""
        print(f"\nScanning for shortcuts in: {self.source_path}")
        
        for root, dirs, files in os.walk(self.source_path):
            for file in files:
                if file.lower().endswith('.lnk'):
                    lnk_path = Path(root) / file
                    
                    # Extract target path from shortcut
                    target_path = self.get_shortcut_target(lnk_path)
                    
                    if target_path and target_path.exists():
                        # Find modified versions
                        modified_files = self.find_modified_versions(target_path)
                        
                        # Store data
                        self.shortcut_data.append({
                            'shortcut_path': lnk_path,
                            'original_path': target_path,
                            'modified_files': modified_files,
                            'folder': root,
                            'shortcut_name': file[:-4]  # Remove .lnk
                        })
        
        print(f"Found {len(self.shortcut_data)} valid shortcuts")
        return self.shortcut_data
    
    def get_shortcut_target(self, lnk_path):
        """Extract target path from Windows shortcut."""
        try:
            pythoncom.CoInitialize()
            shell = win32com.client.Dispatch("WScript.Shell")
            shortcut = shell.CreateShortCut(str(lnk_path))
            target_path = shortcut.TargetPath
            pythoncom.CoUninitialize()
            return Path(target_path) if target_path else None
        except Exception as e:
            print(f"Error reading shortcut {lnk_path.name}: {e}")
            return None
    
    def find_modified_versions(self, original_path):
        """Look for modified versions of the original image."""
        folder = original_path.parent
        stem = original_path.stem
        modified_patterns = ['_modified', '_modifiziert', '_georeferenced', '_processed']
        modified_files = []
        
        # Check for exact pattern matches
        for pattern in modified_patterns:
            potential_file = folder / f"{stem}{pattern}{original_path.suffix}"
            if potential_file.exists():
                modified_files.append(potential_file)
        
        # Check for any file containing modification patterns
        for file in folder.iterdir():
            if file.is_file() and original_path.suffix == file.suffix:
                file_stem = file.stem
                if (file_stem.startswith(stem) and 
                    any(pattern in file_stem for pattern in modified_patterns)):
                    modified_files.append(file)
        
        return list(set(modified_files))  # Remove duplicates
    
    def resample_georeferenced_image(self, input_path, output_path):
        """Resample a georeferenced image to target pixel size."""
        try:
            with rasterio.open(input_path) as src:
                if src.crs is None:
                    print(f"  Skipping {input_path.name} - not georeferenced")
                    return False
                
                # Calculate new dimensions and transform
                current_res_x = abs(src.transform.a)
                current_res_y = abs(src.transform.e)
                
                # Calculate scale factors
                scale_x = current_res_x / self.target_pixel_size
                scale_y = current_res_y / self.target_pixel_size
                
                new_width = int(src.width * scale_x)
                new_height = int(src.height * scale_y)
                
                # Calculate new transform
                left = src.bounds.left
                top = src.bounds.top
                new_transform = rasterio.transform.from_origin(
                    left, 
                    top, 
                    self.target_pixel_size,
                    self.target_pixel_size
                )
                
                # Prepare output metadata
                out_meta = src.meta.copy()
                out_meta.update({
                    'driver': 'GTiff',
                    'height': new_height,
                    'width': new_width,
                    'transform': new_transform,
                    'compress': 'lzw'  # Add compression to save space
                })
                
                # Create array for resampled data
                resampled_data = np.zeros(
                    (src.count, new_height, new_width),
                    dtype=src.dtypes[0]
                )
                
                # Resample each band
                for band_idx in range(src.count):
                    # Read band data
                    band_data = src.read(band_idx + 1)
                    
                    # Reproject/resample the band
                    reproject(
                        source=band_data,
                        destination=resampled_data[band_idx],
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=new_transform,
                        dst_crs=src.crs,
                        resampling=Resampling.bilinear,
                        num_threads=4  # Use multiple threads for speed
                    )
                
                # Write output
                with rasterio.open(output_path, 'w', **out_meta) as dst:
                    dst.write(resampled_data)
                
                print(f"  ✓ Resampled: {output_path.name}")
                print(f"    Original: {src.width}x{src.height} @ {current_res_x:.3f}m")
                print(f"    New: {new_width}x{new_height} @ {self.target_pixel_size:.3f}m")
                return True
                
        except Exception as e:
            print(f"  ✗ Error resampling {input_path.name}: {e}")
            return False
    
    def process_all_images(self):
        """Find all georeferenced images and resample them."""
        print(f"\nProcessing images with target pixel size: {self.target_pixel_size}m")
        print("-" * 60)
        
        processed_count = 0
        skipped_count = 0
        
        for shortcut in self.shortcut_data:
            if not shortcut['modified_files']:
                print(f"No modified versions found for: {shortcut['shortcut_name']}")
                skipped_count += 1
                continue
            
            for img_path in shortcut['modified_files']:
                if img_path.suffix.lower() in ['.tif', '.tiff']:
                    # Check if it's georeferenced
                    try:
                        with rasterio.open(img_path) as src:
                            if src.crs is None:
                                print(f"Skipping {img_path.name} - not georeferenced")
                                continue
                    except:
                        continue
                    
                    # Create output filename
                    base_name = img_path.stem
                    output_filename = f"{base_name}_{self.target_pixel_size}m_resampled.tif"
                    output_path = self.output_dir / output_filename
                    
                    # Skip if already processed
                    if output_path.exists():
                        print(f"Skipping {output_filename} - already exists")
                        continue
                    
                    # Resample the image
                    if self.resample_georeferenced_image(img_path, output_path):
                        processed_count += 1
        
        print("-" * 60)
        print(f"Processing complete!")
        print(f"Resampled: {processed_count} images")
        print(f"Skipped: {skipped_count} shortcuts with no modified images")
        return processed_count
    
    def run(self):
        """Main execution method."""
        print("\n" + "=" * 70)
        print("IMAGE RESAMPLER")
        print("=" * 70)
        
        # Find and parse shortcuts
        self.find_and_parse_shortcuts()
        
        # Process images
        if self.shortcut_data:
            processed = self.process_all_images()
            print(f"\nAll images saved to: {self.output_dir}")
            return processed
        else:
            print("No valid shortcuts found!")
            return 0


# Main execution
if __name__ == "__main__":
    print("\nIMAGE RESAMPLING TOOL")
    print("=" * 40)
    
    source_path = input("Enter source path to scan for shortcuts: ").strip()
    while not Path(source_path).exists():
        print("Path not found! Please try again.")
        source_path = input("Enter source path: ").strip()
    
    output_path = input("Enter output directory path: ").strip()
    Path(output_path).mkdir(parents=True, exist_ok=True)
    
    while True:
        try:
            pixel_size = float(input("Enter target pixel size in meters (e.g., 0.5): ").strip())
            if pixel_size > 0:
                break
            else:
                print("Pixel size must be positive!")
        except ValueError:
            print("Please enter a valid number!")
    
    # Run the resampler
    resampler = ImageResampler(source_path, output_path, pixel_size)
    resampler.run()
    
    print("\n" + "=" * 40)
    print("Processing completed!")


IMAGE RESAMPLING TOOL

IMAGE RESAMPLER

Scanning for shortcuts in: D:\TemporaryID
Found 232 valid shortcuts

Processing images with target pixel size: 15.0m
------------------------------------------------------------
  ✓ Resampled: 3147#00579#106G_2874#ID1767#13_09_1944_modified_15.0m_resampled.tif
    Original: 13088x11532 @ 0.213m
    New: 185x163 @ 15.000m
  ✓ Resampled: 3148#00579#106G_2874#ID1767#13_09_1944_modified_15.0m_resampled.tif
    Original: 13114x11557 @ 0.214m
    New: 187x164 @ 15.000m
  ✓ Resampled: 3149#00579#106G_2874#ID1767#13_09_1944_modified_15.0m_resampled.tif
    Original: 13768x12164 @ 0.213m
    New: 195x172 @ 15.000m
  ✓ Resampled: 3150#00579#106G_2874#ID1767#13_09_1944_modified_15.0m_resampled.tif
    Original: 13924x12527 @ 0.214m
    New: 198x179 @ 15.000m
  ✓ Resampled: 3151#00579#106G_2874#ID1767#13_09_1944_modified_15.0m_resampled.tif
    Original: 14110x12859 @ 0.218m
    New: 205x187 @ 15.000m
  ✓ Resampled: 3152#00579#106G_2874#ID1767#13_09_1944_mo